# Some correlation analyses based off of the google code

In [ ]:
import matplotlib.pyplot as plt
import json
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:

root = Path("C:\\Users\\Hzaab\\SDP-Sequential-Diagnosis\\data\\full")

train_df = pd.read_csv(r'C:\Users\Hzaab\SDP-Sequential-Diagnosis\Casande-RL\Baselines\diaformer\release_train_patients.zip')

test_df = pd.read_csv(r'C:\Users\Hzaab\SDP-Sequential-Diagnosis\Casande-RL\Baselines\diaformer\release_test_patients.zip')

evidences = r'C:\Users\Hzaab\SDP-Sequential-Diagnosis\data\release_evidences.json'

train_out = root / "train_processed.csv"
test_out = root / "test_processed.csv"

train_df = train_df
test = test_df

with open(evidences, "r", encoding="utf-8") as f:
    evidence_info = json.load(f)



Train shape: (1025602, 226)
Test shape: (134529, 226)
Saved: C:\Users\Hzaab\SDP-Sequential-Diagnosis\data\full\train_processed.csv
Saved: C:\Users\Hzaab\SDP-Sequential-Diagnosis\data\full\test_processed.csv
   AGE SEX                PATHOLOGY  \
0   18   M                     URTI   
1   21   M  HIV (initial infection)   
2   19   F                Pneumonia   
3   34   F                     URTI   
4   36   M                     URTI   

   Do you have a fever (either felt or measured with a thermometer)?  \
0                                                  1                   
1                                                  1                   
2                                                  0                   
3                                                  0                   
4                                                  0                   

  Do you feel pain somewhere?  \
0                   temple(L)   
1                   temple(L)   
2     posterior chest wall

In [ ]:

feature_names = {}
used_names = {}

for code, info in evidence_info.items():
    name = (
        info.get("question_en")
        or info.get("name")
        or code
    )
    name = name.strip()
    if name in used_names:
        name = f"{name} [{code}]"

    used_names[name] = code
    feature_names[code] = name
    


In [ ]:

def parse_evidences(value):
    """
    Convert a cell such as:
      ['E_91', 'E_54_@_V_161']
    into:
      {'E_91': None, 'E_54': 'V_161'}
    """
    if pd.isna(value):
        return {}

    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return {}

    result = {}

    for item in value:
        item = str(item).strip()

        if "_@_" in item:
            code, val = item.split("_@_", 1)
            result[code] = val
        else:
            result[item] = None

    return result


def decode_evidence(code, value):
    info = evidence_info[code]
    dtype = str(info.get("data_type", "")).upper()
    default = info.get("default_value", 0)
    meanings = info.get("value_meaning", {}) or {}

    # Binary: presence means positive; absence is handled
    # separately using the evidence's default value.
    if dtype == "B":
        return 1

    if value is None:
        value = default

    # Numerical: prefer an actual numeric value.
    if dtype == "N":
        try:
            return float(value)
        except (TypeError, ValueError):
            meaning = meanings.get(str(value), {})
            text = meaning.get("en", str(value))

            try:
                return float(text)
            except (TypeError, ValueError):
                return np.nan

    # Categorical: translate codes to their English meanings.
    if dtype == "C":
        meaning = meanings.get(str(value), {})
        if isinstance(meaning, dict):
            return meaning.get("en", str(value))
        return str(value)

    # Fallback for any other data types.
    meaning = meanings.get(str(value), {})
    if isinstance(meaning, dict):
        return meaning.get("en", str(value))

    return str(value)


def transform_dataset(df):
    df = df.copy()

    # Required non-evidence columns.
    base_cols = ["AGE", "SEX", "PATHOLOGY"]
    missing = [c for c in base_cols + ["EVIDENCES"] if c not in df.columns]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Remove unwanted columns and retain the target.
    result = df[base_cols].copy()

    # Parse all evidence records.
    parsed = df["EVIDENCES"].apply(parse_evidences)

    # Use every evidence feature found in the JSON definition.
    for code, info in evidence_info.items():
        col = feature_names[code]
        default = info.get("default_value", 0)

        # Set appropriate baseline for absent evidence.
        if str(info.get("data_type", "")).upper() == "B":
            default = int(default) if str(default).isdigit() else 0
        elif str(info.get("data_type", "")).upper() == "N":
            try:
                default = float(default)
            except (TypeError, ValueError):
                default = np.nan
        elif str(info.get("data_type", "")).upper() == "C":
            meaning = (info.get("value_meaning", {}) or {}).get(
                str(default), {}
            )
            if isinstance(meaning, dict):
                default = meaning.get("en", str(default))

        values = []

        for record in parsed:
            if code not in record:
                values.append(default)
            else:
                values.append(decode_evidence(code, record[code]))

        result[col] = values

    return result


In [ ]:

train_processed = transform_dataset(train_df)
test_processed = transform_dataset(test)

test_processed = test_processed.reindex(
    columns=train_processed.columns
)

train_processed.to_csv(train_out, index=False)
test_processed.to_csv(test_out, index=False)

print("Train shape:", train_processed.shape)
print("Test shape:", test_processed.shape)
print("Saved:", train_out)
print("Saved:", test_out)
print(train_processed.head())

In [ ]:
train_df = train_processed
test_df = test_processed
# Check if there are any nans values
print(train_df.columns[train_df.isna().any()].tolist())
print(test_df.columns[test_df.isna().any()].tolist())

[]
[]


In [ ]:
print(train_df['Characterize your pain:'].isna().sum())
print(train_df['What color is the rash?'].isna().sum())
print(test_df['Characterize your pain:'].isna().sum())
print(test_df['What color is the rash?'].isna().sum())

0
0
0
0


In [ ]:
print(f'Train dataset shape: {train_df.shape}')
print(f'Test dataset shape: {test_df.shape}')

'Train dataset shape: (1025602, 226)'
'Test dataset shape: (134529, 226)'


In [ ]:
print(f'Train dataset shape: {train_df.shape}')
print(f'Test dataset shape: {test_df}')

'Train dataset shape: (1025602, 226)'
'Test dataset shape: (134529, 226)'


In [ ]:
# Handling Categorical Variables  - Label Encoding to avoid creating new features that increase the already many featureset
from sklearn.preprocessing import LabelEncoder
# View values of categorical columns
unique_values = {col: train_df[col].unique() for col in train_df.columns if train_df[col].dtype == 'O'}

categorical_columns = list(unique_values.keys())

# Reset indices of both X_train and X_test
train_df_reset = train_df.reset_index(drop=True)
test_df_reset = test_df.reset_index(drop=True)

# Combine the train_df and test data
combined = pd.concat([train_df_reset, test_df_reset], axis=0)

# Create a dictionary to store encodings
label_encoders = {}


# Label encode the combined dataframe for each categorical column
for col in categorical_columns:
  le = LabelEncoder()
  combined[col] = le.fit_transform(combined[col])
  label_encoders[col] = le

combined_encoded = combined

# Split the combined_encoded dataframe back into train and test
train_encoded = combined_encoded.iloc[:len(train_df)]
test_encoded = combined_encoded.iloc[len(train_df):]

In [ ]:
# Prepare training and test variables
X_columns = [col for col in train_df.columns if col not in ['Unnamed: 0', 'PATHOLOGY']]
X_train = train_encoded[X_columns]
y_train = train_encoded['PATHOLOGY']
X_test = test_encoded[X_columns]
y_test = test_encoded['PATHOLOGY']
print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_test shape: {y_test.shape}')


'X_train shape: (1025602, 225)'
'y_train shape: (1025602,)'
'X_test shape: (134529, 225)'
'y_test shape: (134529,)'


In [ ]:
# Check dimension after encoding
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')

'X_train shape: (1025602, 225)'
'X_test shape: (134529, 225)'


In [26]:
# Correlation Analysis
correlation_matrix = train_encoded.corr()
correlation_matrix.style.background_gradient(cmap='cool')

In [ ]:
target_column = "PATHOLOGY"  # Replace with your actual label column name
feature_correlations = correlation_matrix[target_column].sort_values(
    ascending=False
)


In [28]:
feature_correlations

PATHOLOGY                                                                            1.000000
Have you been in contact with a person with similar symptoms in the past 2 weeks?    0.446636
Do you live with 4 or more people?                                                   0.327659
Do you attend or work in a daycare?                                                  0.324909
Have you been coughing up blood?                                                     0.291365
                                                                                       ...   
Have you ever had a diagnosis of anemia?                                            -0.221431
Have you noticed a wheezing sound when you exhale?                                  -0.221831
Have you had a cold in the last 2 weeks?                                            -0.222581
Are you taking any new oral anticoagulants ((NOACs)?                                -0.224602
Are you more likely to develop common allergies than the gen

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ast
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
root = Path("C:\\Users\\Hzaab\\SDP-Sequential-Diagnosis\\data\\full")

train_out = root / "train_processed.csv"
train_processed= pd.read_csv(train_out)


C:\Users\Hzaab\AppData\Local\Temp\ipykernel_23924\1283952869.py:14: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  train_processed= pd.read_csv(TRAIN_OUT)


In [ ]:
import pandas as pd
import numpy as np

CSV_PATH = train_out
TARGET = "PATHOLOGY"
CHUNK_SIZE = 20_000

# First pass: identify diagnoses and numeric feature columns
diagnoses = set()
feature_cols = None

for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE):
    diagnoses.update(chunk[TARGET].dropna().unique())

    if feature_cols is None:
        if "SEX" in chunk.columns:
            chunk["SEX"] = chunk["SEX"].map({"M": 1.0, "F": 0.0})

        feature_cols = chunk.drop(columns=[TARGET]).select_dtypes(
            include="number"
        ).columns.tolist()

diagnoses = sorted(diagnoses)
print("Diagnoses:", len(diagnoses))
print("Numeric features:", len(feature_cols))

# Running statistics for correlation
# For each diagnosis and feature, track:
# count, sum(x), sum(x²), sum(y), sum(y²), sum(x*y)
stats = {
    d: {
        "n": 0,
        "sx": np.zeros(len(feature_cols), dtype=np.float64),
        "sx2": np.zeros(len(feature_cols), dtype=np.float64),
        "sy": 0.0,
        "sy2": 0.0,
        "sxy": np.zeros(len(feature_cols), dtype=np.float64),
    }
    for d in diagnoses
}

# Second pass: process one chunk at a time
for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE):
    if "SEX" in chunk.columns:
        chunk["SEX"] = chunk["SEX"].map({"M": 1.0, "F": 0.0})

    X = chunk[feature_cols].to_numpy(dtype=np.float64)
    y_all = chunk[TARGET].to_numpy()

    for d in diagnoses:
        mask = (y_all == d)
        if not mask.any():
            continue

        # Diagnosis indicator: 1 for this diagnosis, otherwise 0
        Y = mask.astype(np.float64)

        # Skip missing feature values by replacing them with zero
        # after computing sums; assumes features are normally complete.
        valid_X = np.nan_to_num(X, nan=0.0)

        s = stats[d]
        s["n"] += len(Y)
        s["sx"] += valid_X.sum(axis=0)
        s["sx2"] += (valid_X * valid_X).sum(axis=0)
        s["sy"] += Y.sum()
        s["sy2"] += (Y * Y).sum()
        s["sxy"] += valid_X.T @ Y

# Compute correlations
corr = {}

for d, s in stats.items():
    n = s["n"]
    numerator = s["sxy"] - s["sx"] * s["sy"] / n
    var_x = s["sx2"] - s["sx"] ** 2 / n
    var_y = s["sy2"] - s["sy"] ** 2 / n

    denominator = np.sqrt(np.maximum(var_x, 0) * max(var_y, 0))

    values = np.divide(
        numerator,
        denominator,
        out=np.full(len(feature_cols), np.nan),
        where=denominator > 0,
    )

    corr[d] = values

diagnosis_feature_corr = pd.DataFrame(
    corr,
    index=feature_cols
).T

diagnosis_feature_corr.to_csv("diagnosis_feature_correlations.csv")

print("Saved diagnosis_feature_correlations.csv")
print("Matrix shape:", diagnosis_feature_corr.shape)

C:\Users\Hzaab\AppData\Local\Temp\ipykernel_23924\2183898469.py:12: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE):
C:\Users\Hzaab\AppData\Local\Temp\ipykernel_23924\2183898469.py:12: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE):
C:\Users\Hzaab\AppData\Local\Temp\ipykernel_23924\2183898469.py:12: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE):
C:\Users\Hzaab\AppData\Local\Temp\ipykernel_23924\2183898469.py:12: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE):


Diagnoses: 49
Numeric features: 216


C:\Users\Hzaab\AppData\Local\Temp\ipykernel_23924\2183898469.py:43: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE):
C:\Users\Hzaab\AppData\Local\Temp\ipykernel_23924\2183898469.py:43: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE):
C:\Users\Hzaab\AppData\Local\Temp\ipykernel_23924\2183898469.py:43: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE):


Saved diagnosis_feature_correlations.csv
Matrix shape: (49, 216)


C:\Users\Hzaab\AppData\Local\Temp\ipykernel_23924\2183898469.py:43: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE):


In [4]:
for diagnosis in diagnosis_feature_corr.index:
    top = diagnosis_feature_corr.loc[diagnosis].dropna()

    top = top.reindex(
        top.abs().sort_values(ascending=False).head(2).index
    )

    print(f"\n{diagnosis}")
    print(top)


Acute COPD exacerbation / infection
Have you had one or several flare ups of chronic obstructive pulmonary disease (COPD) in the past year?    0.849926
Do you work in the mining sector?                                                                          0.793040
Name: Acute COPD exacerbation / infection, dtype: float64

Acute dystonic reactions
Have you started or taken any antipsychotic medication within the last 7 days?                                                                                              0.837794
Have you been treated in hospital recently for nausea, agitation, intoxication or aggressive behavior and received medication via an intravenous or intramuscular route?    0.836585
Name: Acute dystonic reactions, dtype: float64

Acute laryngitis
Have you noticed that the tone of your voice has become deeper, softer or hoarse?    0.729090
Have you had a cold in the last 2 weeks?                                             0.414887
Name: Acute laryngitis, dtype: f

Acute COPD exacerbation / infection
	
Have you had one or several flare ups of chronic obstructive pulmonary disease (COPD) in the past year?


Acute pulmonary edema
	
Have you ever had fluid in your lungs?


Acute otitis media
	
Are you currently being treated or have you recently been treated with an oral antibiotic for an ear infection?


Pericarditis
	
Have you ever had a pericarditis?


Spontaneous pneumothorax
	
Have you ever had a spontaneous pneumothorax?


Croup
	
Have you or any member of your family ever had croup?

In [ ]:
import pandas as pd

# Load processed datasets
train_df = pd.read_csv(
    r"C:\Users\Hzaab\SDP-Sequential-Diagnosis\data\full\train_processed.csv"
)

test_df = pd.read_csv(
    r"C:\Users\Hzaab\SDP-Sequential-Diagnosis\data\full\test_processed.csv"
)

# Diagnosis and its corresponding question
diagnosis_questions = {
    "Acute COPD exacerbation / infection":
        "Have you had one or several flare ups of chronic obstructive pulmonary disease (COPD) in the past year?",

    "Acute pulmonary edema":
        "Have you ever had fluid in your lungs?",

    "Acute otitis media":
        "Are you currently being treated or have you recently been treated with an oral antibiotic for an ear infection?",

    "Pericarditis":
        "Have you ever had a pericarditis?",

    "Spontaneous pneumothorax":
        "Have you ever had a spontaneous pneumothorax?",

    "Croup":
        "Have you or any member of your family ever had croup?"
}

train_df["Dataset"] = "Train"
test_df["Dataset"] = "Test"

all_df = pd.concat([train_df, test_df], ignore_index=True)

# Check each diagnosis-question pair
results = []

for diagnosis, question in diagnosis_questions.items():

    if question not in all_df.columns:
        print(f"Question column not found: {question}")
        continue

    # Patients with the specified diagnosis
    diagnosis_mask = (
        all_df["PATHOLOGY"].astype(str).str.strip().str.casefold()
        == diagnosis.casefold()
    )

    # Patients whose answer to the question is positive
    answer = all_df[question]

    # Adjust this condition if categorical answers are used
    positive_mask = (
        answer.astype(str).str.strip().str.casefold()
        .isin(["1", "1.0", "true", "yes", "y"])
    )

    # Patients satisfying BOTH conditions
    matched = all_df[diagnosis_mask & positive_mask]

    unmatched = all_df[~diagnosis_mask & positive_mask]

    negative_mask = (
        answer.astype(str).str.strip().str.casefold()
        .isin(["0", "0.0", "false", "no", "n"])
    )

    illogic = all_df[diagnosis_mask & negative_mask]
    results.append({
        "Diagnosis": diagnosis,
        "Question": question,
        "Diagnosis patients": int(diagnosis_mask.sum()),
        "Positive answers": int(positive_mask.sum()),
        "Patients with both": len(matched),
        "Train matches": int((matched["Dataset"] == "Train").sum()),
        "Test matches": int((matched["Dataset"] == "Test").sum()),
        "unmatched patients with positive answer": len(unmatched),
        "diagnosed patients that answered no": len(illogic)
    })

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

# Save the summary
results_df.to_csv(
    "diagnosis_question_matches.csv",
    index=False
)

C:\Users\Hzaab\AppData\Local\Temp\ipykernel_31860\1769406545.py:4: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(


                          Diagnosis                                                                                                        Question  Diagnosis patients  Positive answers  Patients with both  Train matches  Test matches  unmatched patients with positive answer  diagnosed patients that answered no
Acute COPD exacerbation / infection         Have you had one or several flare ups of chronic obstructive pulmonary disease (COPD) in the past year?               19814             14398               14398          12822          1576                                        0                                 5416
              Acute pulmonary edema                                                                          Have you ever had fluid in your lungs?               21616             17886               17886          15726          2160                                        0                                 3730
                 Acute otitis media Are you currently being t